<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 26 · ML for Credit Spreads and Default Risk

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch26_ml_credit_spreads_default_risk.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This chapter turns credit analytics into a supervised classification problem and shows why label design decides what the model can claim. You will learn to build features from spread levels in basis points, spread momentum, leverage, interest coverage, and macro variables; to define labels that do not leak future information; and to fit and inspect a logistic regression under a chronological split. You will evaluate the classifier with precision and recall rather than accuracy, because stress and default labels are rare and the two error types have different costs. You will also practice reading errors issuer by issuer and stating the limits of uncalibrated scores on small synthetic panels.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script fits a logistic regression on the first three weekly snapshots of the synthetic credit panel and holds out the final date, 2026-01-23, so every training label date precedes the test snapshot. It prints a probability and a 0.5-threshold prediction for each held-out issuer, followed by a one-row metrics table. With only three test observations, the result illustrates the workflow rather than providing evidence of model skill.

In [ ]:
SCRIPT = ROOT / "code" / "ch26_ml_credit_spreads_default_risk.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Confirm the split is chronological: the three training snapshots (2026-01-02 to 2026-01-16) all precede the held-out date 2026-01-23, and the script rejects panels with missing or inconsistent timing fields.
Check the label definition in the panel: feature_available_date, date, and label_end_date coincide, so the label is a same-date synthetic stress state, not a realized default or a forecast target.
Trace each test row: probabilities 0.086 (Alpha), 0.883 (Beta), and 0.998 (Gamma) against the 0.5 threshold and the printed predictions.
Treat the perfect metrics (precision, recall, and accuracy all 1.0 on three issuers) as a workflow illustration; three observations cannot support performance claims.
Remember that the printed probabilities are raw, uncalibrated sigmoid scores, not validated default probabilities.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch26_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Logistic stress probability from one ratio feature

The script fits five features at once. This lab isolates the core mechanism: a hand-specified logistic score computed from a single ratio feature, leverage in turns, for three illustrative issuers. The coefficients are fixed by hand, so the probabilities are uncalibrated scores, and the labels are synthetic stress states, not defaults.

In [ ]:
import numpy as np
import pandas as pd

issuers = pd.DataFrame({  # three illustrative issuers
    "issuer": ["Alpha", "Beta", "Gamma"],
    "leverage": [2.5, 4.0, 6.5],  # debt over EBITDA, in turns
    "label": [0, 1, 1],  # stress labels
})
score = -3.0 + 0.8 * issuers.leverage  # hand-set linear score
issuers["probability"] = 1 / (1 + np.exp(-score))  # logistic
for cut in [0.5, 0.6]:  # two thresholds
    hit = issuers.probability.ge(cut)  # predicted stress
    tp = int((hit & issuers.label.eq(1)).sum())
    fn = int((~hit & issuers.label.eq(1)).sum())
    prec = tp / int(hit.sum())  # precision
    print(f"cut {cut}: precision {prec:.2f} recall {tp / (tp + fn):.2f}")
issuers.round(3)

### What to observe

Expected output: probabilities 0.269, 0.550, and 0.900 for Alpha, Beta, and Gamma. At the 0.5 cut all three issuers are classified correctly (precision 1.00, recall 1.00); at 0.6 Beta falls below the cut and recall drops to 0.50 while precision stays 1.00. The threshold, not the score alone, decides the error mix, and none of these values are calibrated probabilities.

## Exercises

1. **Label audit.** Open ch26_credit_ml_panel.csv and confirm that date, feature_available_date, and label_end_date coincide for every row; state in one sentence what this implies about the label. *Hint:* The chapter script raises an error if any timing field is missing or inconsistent.
2. **Threshold sweep.** Re-evaluate the mini-lab probabilities at cuts from 0.3 to 0.7 and tabulate which issuers are flagged at each cut. *Hint:* Only Beta, with a probability of 0.550, changes class inside this range.
3. **Error inspection.** Change the mini-lab intercept so Gamma is also missed at the 0.5 cut, then identify which missed issuer a credit monitoring desk would care about most. *Hint:* Recompute the confusion counts by hand and recall that recall matters most for early-warning screens.
4. **Calibration check.** Explain why a sigmoid output between 0 and 1 is not automatically a calibrated default probability. *Hint:* Calibration compares predicted probabilities with observed frequencies out of sample; nothing in the script performs that comparison.
5. **Feature timing.** Name one credit feature that could leak future information in a real panel and describe the timestamp that would prevent the leak. *Hint:* Think of rating actions or revised financial statements published after the snapshot date.

For more detail, see Chapter 26 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">